# Training-only feature preparation
Only the preprocessing transformer is fitted here. No classifier, threshold, raw record or final result is changed.

In [ ]:
from pathlib import Path
import sys
root = Path.cwd().resolve()
while not (root / 'src' / 'config.py').exists():
    if root.parent == root:
        raise RuntimeError('Open the notebook from the project or notebooks directory')
    root = root.parent
sys.path.insert(0, str(root))
import pandas as pd
from src import config
from src.data_loading import load_master_data
from src.data_validation import add_temporal_label_audit
revision = root / 'outputs/revisions/2026-11-05'

from src.preprocessing import build_preprocessor
frame = add_temporal_label_audit(load_master_data())
train = frame[frame.label_temporally_valid & frame.year.isin(config.TRAIN_YEARS)]
transformer = build_preprocessor()
prepared = transformer.fit_transform(train[config.PREDICTORS])
assert prepared.shape == (2960, 31)
display(pd.DataFrame({'numeric_input':config.NUMERICAL_PREDICTORS,'training_median':transformer.named_transformers_['numeric'].named_steps['imputer'].statistics_}))
print('Prepared training shape:', prepared.shape)